In [96]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import csv
import numpy as np
import copy

In [97]:
df = pd.read_csv(
    "./responses_27mag_PREPROCESS.csv",
    sep=";",
    quoting=csv.QUOTE_MINIMAL,
    encoding="utf-8",
    engine="python"
)

# FEATURE ENGINEERING

mi creo la lista di funzioni che mi vanno a fare la parte di feature engineering e la parte di preprocessing del testo, così che poi posso utilizzare le stesse funzioni nella pipeline per la manipolazione del commento in input

In [98]:
df.head()

,commento,valutazione,tossicita,tipo_tossicita,age_group,gender,utente_id
0,is this a satire subreddit?,Non toxic,0,0,1,1,1
1,Dayum.,Non toxic,0,0,1,1,1
2,And they wouldn’t be scrubbing the floors if t...,Non toxic,0,0,1,1,1
3,Good point. I mean does Nebraska really exist?...,Non toxic,0,0,1,1,1
4,or a yuumi/nami/lulu/zilean/ any enhancer play...,Non toxic,0,0,1,1,1


### 01. EMOJI

In [99]:
import emoji
import emojis
#https://emojis.readthedocs.io/en/latest/index.html

# 1. Applico decode() per convertire le emoji in :emoji_testo
df["emoji_decoded"] = df["commento"].apply(lambda x: emojis.decode(x).replace(":", " "))

# 2. Applico list per ottenere la lista ordinata di tutte le emoji nel commento (con ripetizioni)
df["emoji_list"] = df["commento"].apply(lambda x: [e['emoji'] for e in emoji.emoji_list(x)])

# 3. Applico count() per contare quante emoji ci sono nel commento
df["emoji_count"] = df["commento"].apply(lambda x: emojis.count(x))

# 4. Applico get() per estrarre un set di emoji
df["emoji_set"] = df["commento"].apply(lambda x: emojis.get(x))

### 02. SENTIMENT EMOJI + EMOTAG

In [100]:
# https://www.kaggle.com/datasets/thomasseleck/emoji-sentiment-data?resource=download 

df_sentiment = pd.read_csv(
    "./emoji/Emoji_Sentiment_Data_v1.csv",
    sep=",",
    quoting=csv.QUOTE_MINIMAL,
    encoding="utf-8",
    engine="python"
)

In [101]:
# assegno 0 se è neutrale, 1 se è positivo e -1 se è negativo
def sentiment_label(row):
    if row['Positive'] >= row['Neutral'] and row['Positive'] >= row['Negative']:
        return 1
    elif row['Neutral'] >= row['Positive'] and row['Neutral'] >= row['Negative']:
        return 0
    else:
        return -1

df_sentiment['sentiment_label'] = df_sentiment.apply(sentiment_label, axis=1)

adesso che ho dato uno score alle vari emoji per sentimento (positivo, negativo o neutro), voglio aggiungere anche questo controllo dentro al mio df_emoji 

In [102]:
def emoji_sentiment_score(emoji_list, df_sentiment):
    if not emoji_list:
        return 0

    scores = []
    for emoji_char in emoji_list:
        row = df_sentiment[df_sentiment['Emoji'] == emoji_char]
        if not row.empty:
            scores.append(row['sentiment_label'].values[0])
        else:
            scores.append(0)

    return np.mean(scores) if scores else 0

adesso che ho il voto di sentimento, voglio capire quali sono le emozioni all'interno del commento, facendo una media dei voti di ciascuna emoji per ogni tipologia di emozione qualora ce ne fosse più di una.

!! A volte può essere che nonostante la presenza di emoji, le emozioni non vengano calcolate: questo singifica che la specifica emoji non è contenuta nel file emotag e quindi non è possibile assegnarle uno score.

In [103]:
# https://github.com/abushoeb/EmoTag 
df_emotag = pd.read_csv(
    "./emoji/EmoTag1200-scores.csv",
    sep=",",
    quoting=csv.QUOTE_MINIMAL,
    encoding="utf-8",
    engine="python"
)
df_emotag

,unicode,emoji,name,anger,anticipation,disgust,fear,joy,sadness,surprise,trust
0,1F308,🌈,rainbow,0.00,0.28,0.00,0.00,0.69,0.06,0.22,0.33
1,1F319,🌙,crescent moon,0.00,0.31,0.00,0.00,0.25,0.00,0.06,0.25
2,1F31A,🌚,new moon face,0.06,0.08,0.17,0.06,0.42,0.19,0.06,0.11
3,1F31E,🌞,sun with face,0.00,0.22,0.00,0.00,0.78,0.00,0.11,0.22
4,1F31F,🌟,glowing star,0.00,0.28,0.00,0.00,0.53,0.00,0.25,0.31
...,...,...,...,...,...,...,...,...,...,...,...
145,2757,❗,exclamation mark,0.44,0.42,0.31,0.42,0.08,0.17,0.81,0.11
146,2764,❤,red heart,0.00,0.36,0.00,0.00,0.69,0.00,0.14,0.67
147,27A1,➡,right arrow,0.00,0.06,0.00,0.00,0.00,0.00,0.00,0.22
148,2B05,⬅,left arrow,0.17,0.14,0.17,0.14,0.00,0.14,0.03,0.06


In [104]:
def avg_emoji_emotions(emoji_list):
    matched = df_emotag[df_emotag["emoji"].isin(emoji_list)]
    if matched.empty:
        return pd.Series({col: 0.0 for col in df_emotag.columns if col not in ["unicode", "emoji", "name"]})
    return matched[[col for col in df_emotag.columns if col not in ["unicode", "emoji", "name"]]].mean()

### 03. PAROLE IN MAIUSCOLO

- Se l'intera frase è tutta maiuscola → prendi tutte le parole >= 2 caratteri, escludendo I.
- Se non è tutta maiuscola → prendi solo parole tutte maiuscole che rispettano le regole.
- Non prendere parole capitalizzate (es. Facebook se non è tutta maiuscola).

In [105]:
def trova_maiuscole(testo):
    if not isinstance(testo, str):
        return []
    
    parole = testo.split()
    
    # Caso 1: frase tutta maiuscola, allora prendi tutte le parole >= 2 caratteri (escludendo 'I')
    if testo.isupper():
        return [p for p in parole if len(p) >= 2 and p != "I"]
    
    # Caso 2: frase NON tutta maiuscola, allora prendi solo parole TUTTE maiuscole, >= 2 caratteri, escludendo 'I'
    return [p for p in parole if p.isupper() and len(p) >= 2 and p != "I"]

### 04. PAROLE IN MINUSCOLO

In [106]:
# Ora porto il testo in minuscolo
df["emoji_decoded"] = df["emoji_decoded"].str.lower()

### 05. ESPANDERE CONTRAZIONI

In [107]:
# https://pypi.org/project/contractions/
# !pip install contractions
# !pip install num2words
# !pip install nltk

from html import unescape
import re
import string
import contractions
from num2words import num2words
from nltk.corpus import stopwords


In [108]:
def espandi_contrazioni(testo):
    if isinstance(testo, str):
        return contractions.fix(testo, slang=True)
    return testo

### 06. ELIMINARE PUNTEGGIATURA

In [109]:
punteggiatura = ['"', '$', '%', '&', "'", '(', ')', '*', '+', ',',
            '/', ':', ';', '<', '=', '>', '@', '[', '\\', ']', '^', '_',
            '`', '{', '|', '}', '~', '»', '«', '“', '”', '#', '!', '?', '.', ':']
punct_pattern = re.compile("[" + re.escape("".join(punteggiatura)) + "]")

stop_words = set(stopwords.words("english"))

def text_cleaning(testo):
    if not isinstance(testo, str):
        return testo
    testo = unescape(testo)
    testo = re.sub(r'https?://\S+|www\.\S+', '', testo)
    testo = re.sub(r'[^a-zA-Z0-9\s' + re.escape(string.punctuation) + ']', '', testo)
    testo = re.sub(r'\s+', ' ', testo)
    testo = re.sub(r'[\n\t]', ' ', testo)
    testo = re.sub(punct_pattern, ' ', testo)
    testo = re.sub(r'[0-9\.]+', '', testo) 
    testo = testo.strip()
    
    # Filtra le stopwords
    parole = [word for word in testo.split() if word.lower() not in stop_words]
    
    # Ritorna il testo senza stopwords
    return " ".join(parole)

### 07. NUMERI IN TESTO

In [110]:
# traduzione dei numeri in testo
def numbers_to_text(testo):
    if not isinstance(testo, str):
        return testo
    
    def num_to_text(match):
        numero = int(match.group())
        return num2words(numero, lang="en")
    
    return re.sub(r'\b\d+\b', num_to_text, testo)

### 08. POS TAGGING

In [111]:
import nltk
from nltk.tokenize import word_tokenize

nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')

def posTag(text):
    if not isinstance(text, str):
        return []
    tokens = word_tokenize(text)
    pos_tags = nltk.pos_tag(tokens)
    return pos_tags

[nltk_data] Downloading package punkt to /Users/User/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /Users/User/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


### 09. NRCLex (misura emozioni all'interno di un testo, analizzando ciascuna parola e la frase nel complesso)

The NRC Emotion Lexicon is a list of English words and their associations with eight basic emotions (anger, fear, anticipation, trust, surprise, sadness, joy, and disgust) and two sentiments (negative and positive). The annotations were manually done by crowdsourcing. 

https://saifmohammad.com/WebPages/NRC-Emotion-Lexicon.htm 

In [112]:
# https://pypi.org/project/NRCLex/
# !pip install NRCLex

from nrclex import NRCLex

# Funzione che restituisce tutti i dati da NRCLex
def nrc_emozioni(text):
    if not isinstance(text, str) or text.strip() == "":
        return {
            "nrc_affect_dict": {},
            "nrc_raw_scores": {},
            "nrc_top_emotions": [],
            "nrc_affect_frequencies": {}
        }

    emotion_obj = NRCLex(text)
    
    return {
        "nrc_affect_dict": emotion_obj.affect_dict,  # mappa parola → emozioni
        "nrc_raw_scores": emotion_obj.raw_emotion_scores,  # conteggi grezzi
        "nrc_top_emotions": emotion_obj.top_emotions,  # emozioni dominanti
        "nrc_affect_frequencies": emotion_obj.affect_frequencies  # frequenze normalizzate
    }

### 10. POLARITÀ - TEXTBLOB

TextBlob può fare tante cose, ma come funzionalità di libreria siamo interessati solo alla Sentiment Analysis.

The sentiment property returns a namedtuple of the form Sentiment(polarity, subjectivity). The polarity score is a float within the range [-1.0, 1.0]. The subjectivity is a float within the range [0.0, 1.0] where 0.0 is very objective and 1.0 is very subjective.

La polarità misura se un testo ha un tono positivo, negativo o neutro.

In [113]:
# https://textblob.readthedocs.io/en/dev/quickstart.html

from textblob import TextBlob   

def polarità(text):
    try:
        return TextBlob(text).sentiment.polarity
    except:
        return None

### 11. READABILITY

- <b>Kincaid (Flesch-Kincaid Grade Level)</b>: stima il grado scolastico minimo per comprendere il testo (es. 5.44 ≈ 5ª classe).

- <b>ARI (Automated Readability)</b>:: misura la difficoltà di lettura basata su parole e frasi; restituisce il grado scolastico richiesto.

- <b>Coleman-Liau</b>:: stima il grado scolastico usando lunghezza media delle parole e frasi (usa caratteri, non sillabe).

- F<b>lesch Reading Ease</b>: scala da 0 a 100, dove valori alti = testo più facile da leggere (es. >80 = molto facile).

- <b>Gunning Fog</b>: stima il livello scolastico considerando le parole complesse (più alto = più difficile).

- <b>LIX</b>: indicatore svedese che misura la leggibilità basandosi su lunghezza delle frasi e parole lunghe.

- <b>SMOG</b>: calcola gli anni di istruzione necessari per capire un testo basandosi sulle parole polysyllabic.

- <b>RIX (Readability)</b>: misura simile al LIX, basata su parole lunghe per frase (valori più alti = testo più difficile).

- <b>Dale-Chall</b>: valuta la leggibilità confrontando le parole con una lista di parole “facili” e contando le difficili

In [114]:
# https://pypi.org/project/readability/0.3.2/

#!pip install https://github.com/andreasvc/readability/tarball/master
#!pip install syntok

import readability
import syntok.segmenter as segmenter

def text_readability(text):
    try:
        tokenized = '\n\n'.join(
            '\n'.join(' '.join(token.value for token in sentence)
                      for sentence in paragraph)
            for paragraph in segmenter.analyze(text)
        )
        scores = readability.getmeasures(tokenized, lang='en')
        return scores['readability grades']
    except:
        return {}

### 12. TOKENIZATION (divido la frase in singole parole, oppure creo bigrammi e trigrammi --> utile per trovare parolacce o parole composte)

In [115]:
import re
import nltk

def tokenization(text):
    if not isinstance(text, str):
        return []
    return re.split(r'\W+', text.lower().strip())

### 13. LEMMATIZATION (riporto le parole alla forma base del verbo o sostantivo)

In [116]:
from nltk.stem import WordNetLemmatizer

wordlemm = WordNetLemmatizer()

def lemmatization(data, wordlemm):    
    new_words_lemm=[]    
    for row in data:        
        new_words_lemm_tmp=[]        
        for word in row:            
            lemm_word = wordlemm.lemmatize(word)            
            lemm_word = wordlemm.lemmatize(lemm_word, pos='v')            
            new_words_lemm_tmp.append(lemm_word)        
        new_words_lemm.append(new_words_lemm_tmp)
    return new_words_lemm

### 14. NRC-VAD (Valence, Arousal and Dominance)

Diversi studi influenti di analisi fattoriale hanno dimostrato che le dimensioni primarie del significato della parola sono la valenza, l'eccitazione e la dominanza (VAD)

- La valenza è la dimensione positiva - negativa o di piacere - dispiacere;
- l'eccitazione è la dimensione passiva eccitata - calma o attiva; e
- il dominio è la dimensione potente - debole o "avere il pieno controllo" - "non avere alcun controllo".

Il lessico NRC Valence, Arousal, and Dominance (VAD) include un elenco di oltre 20.000 parole inglesi e i loro punteggi di valenza, eccitazione e dominanza. Per una determinata parola e una dimensione (V/A/D), i punteggi vanno da 0 (V/A/D più basso) a 1 (V/A/D più alto).

In [117]:
# https://saifmohammad.com/WebPages/nrc-vad.html 

VAD = pd.read_csv('./NRC-VAD/NRC-VAD-Lexicon-v2.1.txt', sep="\t", header=None)
VAD.columns = ["term", "valence", "arousal", "dominance"]
VAD = VAD.dropna(subset=["term"]).copy()
VAD["term"] = VAD["term"].astype(str).str.strip().str.lower()

for c in ["valence", "arousal", "dominance"]:
    VAD[c] = pd.to_numeric(VAD[c], errors="coerce")

VAD = VAD.dropna(subset=["valence", "arousal", "dominance"])
VAD_dict = VAD.set_index("term")[["valence", "arousal", "dominance"]].to_dict("index")

In [118]:
def emotion_VAD(tokens, dim):
    score = []
    seen = set()  # evita conteggi doppi

    for token in tokens:
        # Se è un bigramma (tuple) -> unisce in stringa
        if isinstance(token, tuple):
            token = ' '.join(token)
        token = str(token).strip().lower()
        used_any = False

        # Prova frase intera
        if token in VAD_dict and token not in seen:
            val = VAD_dict[token].get(dim, 0)
            if pd.notna(val):
                score.append(float(val))
                seen.update(token.split())  # segna parole componenti
                used_any = True

        # Fallback su parole singole
        if not used_any:
            for sub_token in token.split():
                sub_token = sub_token.strip().lower()
                if sub_token in VAD_dict and sub_token not in seen:
                    val = VAD_dict[sub_token].get(dim, 0)
                    if pd.notna(val):
                        score.append(float(val))
                        seen.add(sub_token)

    return sum(score) / len(score) if score else 0.0


def analyze_valence(tokens):   
    return emotion_VAD(tokens, 'valence')

def analyze_arousal(tokens):   
    return emotion_VAD(tokens, 'arousal')

def analyze_dominance(tokens): 
    return emotion_VAD(tokens, 'dominance')

### 15. PAROLACCE / BAD WORDS

controllo se nel testo tokenizzato oppure in quello a bi/trigrammi ci sono delle parolacce e mi salvo il flag in una colonna 

In [119]:
# https://github.com/LDNOOBW/List-of-Dirty-Naughty-Obscene-and-Otherwise-Bad-Words/blob/master/en

# per semplicità ho scaricato la lista di parole in un file .txt

with open("./bad_words_en.txt", "r", encoding="utf-8") as f:
    bad_words = [line.strip().lower() for line in f if line.strip()]

def bad_words_info(tokens):
    text_ngrams = tokens[:]  # unigrammi
    text_ngrams += [' '.join(tokens[i:i+2]) for i in range(len(tokens)-1)]  # bigrammi
    text_ngrams += [' '.join(tokens[i:i+3]) for i in range(len(tokens)-2)]  # trigrammi
    matches = [ng for ng in text_ngrams if ng in bad_words]
    flag = int(len(matches) > 0)
    count = len(matches)
    return pd.Series([flag, count, ', '.join(matches)])

### APPLICAZIONE DELLE FUNZIONI

In [121]:
def decode_emojis(df, col):
    df["emoji_decoded"] = df[col].apply(lambda x: emojis.decode(x).replace(":", " "))
    df["emoji_list"] = df[col].apply(lambda x: [e['emoji'] for e in emoji.emoji_list(x)])
    df["emoji_count"] = df[col].apply(lambda x: emojis.count(x))
    df["emoji_set"] = df[col].apply(lambda x: emojis.get(x))
    return df

def add_emoji_sentiment(df):
    df['emoji_sentiment_score'] = df['emoji_list'].apply(lambda x: emoji_sentiment_score(x, df_sentiment))
    emotion_scores = df["emoji_list"].apply(avg_emoji_emotions)
    df = pd.concat([df, emotion_scores], axis=1)
    return df

def add_upper_words(df):
    df["maiuscolo"] = df["emoji_decoded"].apply(trova_maiuscole)
    return df

def lower_case(df):
    df["emoji_decoded"] = df["emoji_decoded"].str.lower()
    return df

def expand_contractions(df):
    df["emoji_decoded"] = df["emoji_decoded"].apply(espandi_contrazioni)
    return df

def clean_text(df):
    df["emoji_decoded"] = df["emoji_decoded"].apply(text_cleaning)
    return df

def numbers_to_text_df(df):
    df["emoji_decoded"] = df["emoji_decoded"].apply(numbers_to_text)
    return df

def add_pos_tag(df):
    df["pos_tag"] = df["emoji_decoded"].apply(posTag)
    return df

def add_nrc(df):
    nrc_details = df["emoji_decoded"].apply(nrc_emozioni)
    nrc_expanded = pd.DataFrame(nrc_details.tolist())
    df = pd.concat([df, nrc_expanded], axis=1)
    return df

def add_readability(df):
    readb = df["emoji_decoded"].apply(text_readability)
    readability_df = pd.DataFrame(readb.tolist())
    df = pd.concat([df, readability_df], axis=1)
    return df

def add_polarity(df):
    df["polarita"] = df["emoji_decoded"].apply(polarità)
    return df

def add_tokenization(df):
    df['tokenized'] = df['emoji_decoded'].apply(tokenization)
    df['token=2'] = df['tokenized'].apply(lambda words: list(nltk.ngrams(words, 2)))
    df['token=3'] = df['tokenized'].apply(lambda words: list(nltk.ngrams(words, 3)))
    return df

def add_lemmatization(df):
    df["lemmatized"] = lemmatization(df["tokenized"], wordlemm)
    return df

def add_vad(df):
    df["VAD_valence"]   = df["token=2"].apply(analyze_valence)
    df["VAD_arousal"]   = df["token=2"].apply(analyze_arousal)
    df["VAD_dominance"] = df["token=2"].apply(analyze_dominance)
    return df

def add_bad_words(df):
    df[['badwords_flag', 'badwords_count', 'badwords_matches']] = df['tokenized'].apply(bad_words_info)
    return df

In [122]:
def feature_pipeline(df, col):
    df = decode_emojis(df, col)
    df = add_emoji_sentiment(df)
    df = add_upper_words(df)
    df = lower_case(df)
    df = expand_contractions(df)
    df = clean_text(df)
    df = numbers_to_text_df(df)
    df = add_pos_tag(df)
    df = add_nrc(df)
    df = add_readability(df)
    df = add_polarity(df)
    df = add_tokenization(df)
    df = add_lemmatization(df)
    df = add_vad(df)
    df = add_bad_words(df)
    return df

In [123]:
# applico a questo specifico df
df = feature_pipeline(df, "commento")

In [125]:
df

,commento,valutazione,tossicita,tipo_tossicita,age_group,gender,utente_id,emoji_decoded,emoji_list,emoji_count,...,tokenized,token=2,token=3,lemmatized,VAD_valence,VAD_arousal,VAD_dominance,badwords_flag,badwords_count,badwords_matches
0,is this a satire subreddit?,Non toxic,0,0,1,1,1,satire subreddit,[],0,...,"[satire, subreddit]","[(satire, subreddit)]",[],"[satire, subreddit]",-0.084000,0.310000,-0.056000,0,0,
1,Dayum.,Non toxic,0,0,1,1,1,dayum,[],0,...,[dayum],[],[],[dayum],0.000000,0.000000,0.000000,0,0,
2,And they wouldn’t be scrubbing the floors if t...,Non toxic,0,0,1,1,1,would scrubbing floors employed,[],0,...,"[would, scrubbing, floors, employed]","[(would, scrubbing), (scrubbing, floors), (flo...","[(would, scrubbing, floors), (scrubbing, floor...","[would, scrub, floor, employ]",-0.102000,0.282500,-0.358500,0,0,
3,Good point. I mean does Nebraska really exist?...,Non toxic,0,0,1,1,1,good point mean nebraska really exist ever met...,[],0,...,"[good, point, mean, nebraska, really, exist, e...","[(good, point), (point, mean), (mean, nebraska...","[(good, point, mean), (point, mean, nebraska),...","[good, point, mean, nebraska, really, exist, e...",0.326000,-0.267333,0.086333,0,0,
4,or a yuumi/nami/lulu/zilean/ any enhancer play...,Non toxic,0,0,1,1,1,yuumi nami lulu zilean enhancer plays team,[],0,...,"[yuumi, nami, lulu, zilean, enhancer, plays, t...","[(yuumi, nami), (nami, lulu), (lulu, zilean), ...","[(yuumi, nami, lulu), (nami, lulu, zilean), (l...","[yuumi, nami, lulu, zilean, enhancer, play, team]",0.687000,0.294500,0.521500,0,0,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5270,Commies are going to commie. These people hate...,Extremely violent,3,4,1,0,211,commies going commie people hate want die,[],0,...,"[commies, going, commie, people, hate, want, die]","[(commies, going), (going, commie), (commie, p...","[(commies, going, commie), (going, commie, peo...","[commie, go, commie, people, hate, want, die]",-0.329714,0.375857,0.045000,0,0,
5271,Yeah but then the good guys win and rape you t...,Extremely violent,3,4,1,0,211,yeah good guys win rape death occupy nation in...,[],0,...,"[yeah, good, guys, win, rape, death, occupy, n...","[(yeah, good), (good, guys), (guys, win), (win...","[(yeah, good, guys), (good, guys, win), (guys,...","[yeah, good, guy, win, rape, death, occupy, na...",0.012750,0.015250,0.121125,1,1,rape
5272,If you're white and use your gun to defend you...,A little violent,1,4,1,0,211,white use gun defend arrested racist get life ...,[],0,...,"[white, use, gun, defend, arrested, racist, ge...","[(white, use), (use, gun), (gun, defend), (def...","[(white, use, gun), (use, gun, defend), (gun, ...","[white, use, gun, defend, arrest, racist, get,...",-0.115429,0.365714,0.212857,0,0,
5273,I will rape Elon Musk,A little violent,1,4,1,0,211,rape elon musk,[],0,...,"[rape, elon, musk]","[(rape, elon), (elon, musk)]","[(rape, elon, musk)]","[rape, elon, musk]",-0.469000,0.310000,-0.020000,1,1,rape


### SALVATAGGIO IN UN FILE CSV

In [124]:
df.to_csv("./feature_engineering_survey.csv", index=False, encoding='utf-8', sep=';')